# 1.14 如何接著寫下去？


學習時每道題都有已知答案，寫新文字時卻沒有答案可對照。我們給模型一個開頭，讓它分配下一字的機率，選出一字，接到句尾，再把新字當成下一輪的前文。這樣反覆使用「猜下一字」，就是生成（generation）。模型不是一次拿出整篇文章，而是一步一步延長當前內容。

為了看清流程，本節不訓練參數，直接指定 [1.5](https://birdhackor.github.io/tiny-perceptron-vlm/1.5.html) 那種 bigram 機率表。字表為 `['貓','看','狗','。']`，目前是貓就取第0列，下一字「看」有0.85的機會；目前是看就取第1列，狗0.5、貓0.4。每列獨立加總為1。這張表只看最近一字，所以不保證能維持長句意義。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import torch

chars = ["貓", "看", "狗", "。"]
p = torch.tensor(
    [
        [0.05, 0.85, 0.05, 0.05],
        [0.40, 0.05, 0.50, 0.05],
        [0.05, 0.70, 0.05, 0.20],
        [0.50, 0.05, 0.40, 0.05],
    ]
)
g = torch.Generator().manual_seed(42)
current = 0
output = [chars[current]]
for _ in range(12):
    current = torch.multinomial(p[current], 1, generator=g).item()
    output.append(chars[current])
print("".join(output))
assert len(output) == 13

輸入是這張固定表與起始ID0，也就是貓。tensor（張量）是 PyTorch 存放數字的容器；這裡的 `p` 裝一張機率表，抽樣結果則只裝一個 ID。`torch.multinomial` 按給定的機率隨機抽候選，第二個輸入1表示抽一個；回傳 tensor，再用 `.item()` 取出普通整數，成為新 `current`。下一行把它對應的字加入 `output`。`append` 是清單的「在尾端增加一項」方法。第一次從貓那列抽，下一次究竟查哪列，要由剛才抽中的字決定，而不是一直查貓。

`for _ in range(12)` 重複十二次；`range(12)` 提供十二個編號，`_` 表示我們不需要使用那個編號。開頭已經存一字，所以結尾長度是十三。`join` 將清單接回文字，解碼概念可回看 [1.1](https://birdhackor.github.io/tiny-perceptron-vlm/1.1.html)。可以有「貓看狗」這樣的片段，也可能抽到不常見的「貓貓」或連續句號，因為它們的機率不是零。

`torch.Generator` 是本段專用的隨機數工具，`manual_seed(42)` 固定起始狀態。相同PyTorch環境、相同表、相同種子重新從頭執行，通常會重現同一個抽樣序列；更換種子只換抽到的結果，不改變表中的知識。固定種子方便比較實驗，但不能保證跨不同軟體版本與裝置一律完全相同。

這個程式沒有遇到句號就停止，因為這裡的句號是普通候選字。我們明確規定「再抽十二次」才結束。後面做對話會有獨立的結束標記，讓模型有機會自己表示回答完成；別把標點和那種標記混為一談。短句看起來順，也不能據此說這張四字表理解了動物。

換成真的訓練過的接字表，生成流程仍是一字接一字。[T.3的實驗](https://birdhackor.github.io/tiny-perceptron-vlm/training.html#T.3)用12篇「顏色=紅；形狀=圓。」這類短文，拿其中9篇練習200次。訓練後給它開頭`顏色=`，這次每一步選機率最高的一字，它卻接出`三角。`，合起來成了「顏色=三角。」。三角是形狀，沒有完成顏色欄位；這是實際模型的輸出，不是我們指定的機率表。

為什麼會混在一起？「顏色=」和「形狀=」最後都是等號，只看上一字的表在這兩處收到相同輸入，無從知道正在填哪一欄。平均下一字代價可以下降，自己接出來的句子卻仍可能出錯。下一章會把更多前文交給模型，讓它有機會分清這兩種情境；仍要實際訓練與生成，才能知道它用了沒有。

練習只把種子42改成7，先預測哪些東西應保持不變：字表、每列機率、起始貓、總長度十三。重新執行核對這些，再比較文字有沒有不同；個別位置可能恰好相同，是抽樣允許的結果。最後把迴圈次數12改成3，並把檢查長度改成4，核對一次生成只新增一字的規則。
